# 03 - Data Analysis – Tai nghe / Headphone

Notebook này tách riêng phần **phân tích dữ liệu**:

- Đọc dữ liệu đã clean từ `clean_data/headphone_clean.csv`.
- Khảo sát cấu trúc, phân bố giá.
- Phân tích theo hãng, loại tai nghe, gaming / wireless / mic.
- Xem tương quan giữa một số biến số và giá.


## 1. Import thư viện & đọc dữ liệu


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
pd.options.display.float_format = '{:,.0f}'.format

PROJECT_DIR = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
CLEAN_CSV = os.path.join(PROJECT_DIR, 'clean_data', 'headphone_clean.csv')
df = pd.read_csv(CLEAN_CSV)
print('File clean:', CLEAN_CSV)
print('Số dòng:', len(df))
df.head()


## 2. Thông tin tổng quan & thống kê cơ bản


In [ ]:
df.info()

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if numeric_cols:
    display(df[numeric_cols].describe())


### 2.1 Số giá trị phân biệt mỗi cột


In [ ]:
distinct_counts = df.nunique().sort_values(ascending=False)
display(distinct_counts)


## 3. Phân bố giá & phân tích theo hãng / loại


In [ ]:
# Phân bố giá
plt.figure(figsize=(8,4))
sns.histplot(df['price_vnd'], bins=50)
plt.title('Phân bố giá (VND)')
plt.xlabel('Giá (VND)')
plt.ylabel('Số sản phẩm')
plt.show()


In [ ]:
# Boxplot giá theo loại tai nghe
plt.figure(figsize=(6,4))
sns.boxplot(data=df, x='type', y='price_vnd')
plt.title('Giá theo loại tai nghe')
plt.xlabel('Type')
plt.ylabel('Giá (VND)')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Top brand theo median giá (lọc brand xuất hiện đủ nhiều)
brand_counts = df['brand'].value_counts(dropna=True)
popular = brand_counts[brand_counts >= 10].index
tmp = df[df['brand'].isin(popular)].copy()
top = tmp.groupby('brand')['price_vnd'].median().sort_values(ascending=False).head(15)
plt.figure(figsize=(10,5))
sns.barplot(x=top.values, y=top.index)
plt.title('Top 15 hãng theo median giá')
plt.xlabel('Median price (VND)')
plt.ylabel('Brand')
plt.show()


## 4. Gaming / Wireless / Mic và giá


In [ ]:
for col in ['is_gaming', 'is_wireless', 'has_mic']:
    if col in df.columns:
        plt.figure(figsize=(5,4))
        sns.boxplot(data=df, x=col, y='price_vnd')
        plt.title(f'Giá theo {col}')
        plt.xlabel(col)
        plt.ylabel('Giá (VND)')
        plt.show()


## 5. Ma trận tương quan các biến số


In [ ]:
num_cols = ['price_vnd', 'battery_life_hours', 'weight_gram', 'is_gaming', 'is_wireless', 'has_mic']
num_cols = [c for c in num_cols if c in df.columns]

corr = df[num_cols].corr()
plt.figure(figsize=(6,4))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Ma trận tương quan các biến số')
plt.show()
